# 98 — Scaffold-Aware k-NN Ensemble

**Motivation:** Standard ECFP4 Tanimoto k-NN treats all bits equally. PXR analogs share Bemis-Murcko scaffolds, so scaffold-level similarity carries extra signal. Side-chain perturbations (substituent differences) should be captured via physchem distances.

**Strategy:**
1. k-NN system A: ECFP4 Tanimoto (standard)
2. k-NN system B: scaffold-level Tanimoto (Morgan FP of Murcko scaffold SMILES)
3. k-NN system C: physicochemical property distance (normalized Euclidean on MW, logP, TPSA, HBD, HBA, rotbonds, rings, fsp3)
4. Stack OOF predictions from A, B, C via ElasticNet meta-learner
5. Also compare to diversity-blended prediction: `(A + B + C) / 3`

**Key difference from nb05:** Here we scaffold-Morgan-FP the core separately and weight predictions by scaffold similarity, then stack with ElasticNet for the final ensemble.

In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"): sys.stdout.reconfigure(encoding="utf-8")
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import stats
from pathlib import Path
from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, morgan_fp_batch, standardize_smiles, compute_physchem
from pxr.paths import DATA_PROCESSED, DATA_EXTERNAL, SUBMISSIONS
SEED = 42; N_FOLDS = 5
LGBM = dict(n_estimators=1000, num_leaves=64, learning_rate=0.05,
            min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4)
from sklearn.linear_model import ElasticNetCV


In [2]:
def full_metrics(y_true, y_pred, cp=None, label=""):
    yt = np.asarray(y_true, float); yp = np.asarray(y_pred, float)
    msk = np.isfinite(yt) & np.isfinite(yp); yt, yp = yt[msk], yp[msk]
    mae = float(np.mean(np.abs(yt-yp)))
    rae_v = mae / float(np.mean(np.abs(yt-yt.mean()))) if yt.std()>0 else float("nan")
    r2  = 1-np.sum((yt-yp)**2)/np.sum((yt-yt.mean())**2) if yt.std()>0 else float("nan")
    pr, _ = stats.pearsonr(yt, yp); sp, _ = stats.spearmanr(yt, yp)
    kt, _ = stats.kendalltau(yt, yp)
    m = dict(RAE=rae_v, MAE=mae, R2=float(r2), Pearson=float(pr),
             Spearman=float(sp), Kendall=float(kt))
    if cp is not None and hasattr(cp, "iterrows") and len(cp) > 0:
        c=t=0
        for _,row in cp.iterrows():
            ia,ii = int(row.get("idx_active",-1)), int(row.get("idx_inactive",-1))
            if 0<=ia<len(yp) and 0<=ii<len(yp): c+=int(yp[ia]>yp[ii]); t+=1
        m["Cliff_acc"] = c/t if t else float("nan")
    if label:
        ca = f"  Cliff={m.get('Cliff_acc',float('nan')):.3f}" if "Cliff_acc" in m else ""
        print(f"  [{label}] RAE={rae_v:.4f} MAE={mae:.4f} R2={r2:.4f} "
              f"r={pr:.4f} rho={sp:.4f} tau={kt:.4f}{ca}")
    return m


In [3]:
tr = load_train(); te = load_test()
y_tr = tr["pec50"].values.astype(np.float64)
scaffolds = tr["smiles"].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, N_FOLDS, SEED)
active_mask = y_tr >= 5.5
cliff_pairs = (pd.read_parquet(DATA_PROCESSED/"cliff_pairs.parquet")
               if (DATA_PROCESSED/"cliff_pairs.parquet").exists() else pd.DataFrame())
if len(cliff_pairs) > 0:
    s2i = {s:i for i,s in enumerate(tr["smiles"].tolist())}
    ac = "cliff_active_smiles" if "cliff_active_smiles" in cliff_pairs.columns else "smiles_a"
    ic = "cliff_inactive_smiles" if "cliff_inactive_smiles" in cliff_pairs.columns else "smiles_b"
    cliff_pairs["idx_active"]   = cliff_pairs[ac].map(s2i)
    cliff_pairs["idx_inactive"] = cliff_pairs[ic].map(s2i)
    cliff_pairs = cliff_pairs.dropna(subset=["idx_active","idx_inactive"])
    cliff_pairs[["idx_active","idx_inactive"]] = cliff_pairs[["idx_active","idx_inactive"]].astype(int)
print(f"Train {len(tr):,}  Test {len(te):,}  Cliffs {len(cliff_pairs)}")


Train 4,139  Test 513  Cliffs 0


In [4]:
# ---- Feature set A: ECFP4 Morgan FP (2048-bit) ----
print("Computing Morgan FPs...", flush=True)
fps_tr = morgan_fp_batch(tr["smiles"].tolist()).astype(np.float32)
fps_te = morgan_fp_batch(te["smiles"].tolist()).astype(np.float32)

# ---- Feature set B: Scaffold Morgan FP ----
print("Computing scaffold SMILES and scaffold Morgan FPs...", flush=True)
scaf_smiles_tr = tr["smiles"].map(bemis_murcko).tolist()
scaf_smiles_te = te["smiles"].map(bemis_murcko).tolist()
# Replace empty/None scaffold with the compound itself
scaf_smiles_tr = [s if s and len(s) > 0 else tr["smiles"].iloc[i]
                  for i, s in enumerate(scaf_smiles_tr)]
scaf_smiles_te = [s if s and len(s) > 0 else te["smiles"].iloc[i]
                  for i, s in enumerate(scaf_smiles_te)]
scaf_fps_tr = morgan_fp_batch(scaf_smiles_tr).astype(np.float32)
scaf_fps_te = morgan_fp_batch(scaf_smiles_te).astype(np.float32)
print(f"Scaffold FP shape: train={scaf_fps_tr.shape}  test={scaf_fps_te.shape}")

# ---- Feature set C: Physicochemical property vector ----
print("Computing physchem descriptors...", flush=True)
PHYS_PROPS = ["mw","logp","tpsa","hbd","hba","rotbonds","fsp3","rings"]
phys_tr = tr["smiles"].map(compute_physchem).tolist()
phys_te = te["smiles"].map(compute_physchem).tolist()
phys_arr_tr = np.array([[p.get(k,0) or 0 for k in PHYS_PROPS] for p in phys_tr], dtype=np.float32)
phys_arr_te = np.array([[p.get(k,0) or 0 for k in PHYS_PROPS] for p in phys_te], dtype=np.float32)
# Normalize per feature
phys_mean = phys_arr_tr.mean(0); phys_std = phys_arr_tr.std(0) + 1e-6
phys_arr_tr_n = (phys_arr_tr - phys_mean) / phys_std
phys_arr_te_n = (phys_arr_te - phys_mean) / phys_std
print(f"Physchem shape: {phys_arr_tr_n.shape}")


Computing Morgan FPs...


Computing scaffold SMILES and scaffold Morgan FPs...


Scaffold FP shape: train=(4139, 2048)  test=(513, 2048)
Computing physchem descriptors...


Physchem shape: (4139, 8)


In [5]:
# ---- k-NN prediction utility ----
K = 7

def tanimoto_sim(fps_query, fps_ref):
    """Exact Tanimoto similarity: (N_q, N_r) matrix."""
    dot = (fps_query @ fps_ref.T).astype(np.float64)
    rs_q = fps_query.sum(1)[:,None]
    rs_r = fps_ref.sum(1)[None,:]
    union = rs_q + rs_r - dot
    return np.where(union>0, dot/union, 0.0).astype(np.float32)

def physchem_sim(phys_query, phys_ref, gamma=1.0):
    """RBF kernel similarity from normalized Euclidean distance."""
    # (N_q, N_r)
    diff = phys_query[:,None,:] - phys_ref[None,:,:]  # (Nq, Nr, D)
    dist2 = (diff**2).sum(-1)  # (Nq, Nr)
    return np.exp(-gamma * dist2).astype(np.float32)

def knn_predict(sim_matrix, y_ref, k=K):
    """Similarity-weighted k-NN regression."""
    top_k_idx = np.argsort(-sim_matrix, axis=1)[:, :k]
    top_k_sim  = np.take_along_axis(sim_matrix, top_k_idx, axis=1)
    top_k_y    = y_ref[top_k_idx]
    weights = np.maximum(top_k_sim, 1e-8)
    return (weights * top_k_y).sum(1) / weights.sum(1)

print(f"k-NN utilities ready (k={K})")


k-NN utilities ready (k=7)


In [6]:
# ---- Scaffold 5-fold CV ----
print("\n=== Scaffold 5-fold CV ===", flush=True)
oof_knn_a = np.full(len(y_tr), np.nan)  # ECFP4 Tanimoto
oof_knn_b = np.full(len(y_tr), np.nan)  # Scaffold Tanimoto
oof_knn_c = np.full(len(y_tr), np.nan)  # Physchem RBF

for fold, (tr_idx, va_idx) in enumerate(splits):
    fps_va   = fps_tr[va_idx];    fps_ft   = fps_tr[tr_idx]
    scaf_va  = scaf_fps_tr[va_idx]; scaf_ft  = scaf_fps_tr[tr_idx]
    phys_va  = phys_arr_tr_n[va_idx]; phys_ft  = phys_arr_tr_n[tr_idx]
    y_ft     = y_tr[tr_idx]

    # System A: ECFP4
    sim_a = tanimoto_sim(fps_va, fps_ft)
    oof_knn_a[va_idx] = knn_predict(sim_a, y_ft)

    # System B: Scaffold Tanimoto
    sim_b = tanimoto_sim(scaf_va, scaf_ft)
    oof_knn_b[va_idx] = knn_predict(sim_b, y_ft)

    # System C: Physchem RBF
    sim_c = physchem_sim(phys_va, phys_ft, gamma=0.5)
    oof_knn_c[va_idx] = knn_predict(sim_c, y_ft)

    ra = rae(y_tr[va_idx], oof_knn_a[va_idx])
    rb = rae(y_tr[va_idx], oof_knn_b[va_idx])
    rc = rae(y_tr[va_idx], oof_knn_c[va_idx])
    avg_3 = (oof_knn_a[va_idx] + oof_knn_b[va_idx] + oof_knn_c[va_idx]) / 3
    rd = rae(y_tr[va_idx], avg_3)
    print(f"  fold {fold+1}  knnA={ra:.4f}  knnB={rb:.4f}  knnC={rc:.4f}  avg3={rd:.4f}",
          flush=True)

m_a = full_metrics(y_tr, oof_knn_a, cliff_pairs, "knn_ecfp4")
m_b = full_metrics(y_tr, oof_knn_b, cliff_pairs, "knn_scaffold")
m_c = full_metrics(y_tr, oof_knn_c, cliff_pairs, "knn_physchem")
oof_avg3 = (oof_knn_a + oof_knn_b + oof_knn_c) / 3
m_avg = full_metrics(y_tr, oof_avg3, cliff_pairs, "simple_avg3")
print("\n" + pd.DataFrame([m_a, m_b, m_c, m_avg],
                           index=["ecfp4","scaffold","physchem","avg3"]).round(4).to_string())



=== Scaffold 5-fold CV ===


  fold 1  knnA=0.6874  knnB=0.6688  knnC=0.6080  avg3=0.5790


  fold 2  knnA=0.7221  knnB=0.7834  knnC=0.7272  avg3=0.6591


  fold 3  knnA=0.7591  knnB=0.8341  knnC=0.7708  avg3=0.7243


  fold 4  knnA=0.7607  knnB=0.8066  knnC=0.7184  avg3=0.6915


  fold 5  knnA=0.7815  knnB=0.8419  knnC=0.7455  avg3=0.7193


  [knn_ecfp4] RAE=0.7367 MAE=0.6703 R2=0.3926 r=0.6279 rho=0.5799 tau=0.4069
  [knn_scaffold] RAE=0.7795 MAE=0.7093 R2=0.3089 r=0.5634 rho=0.5018 tau=0.3490
  [knn_physchem] RAE=0.7074 MAE=0.6436 R2=0.3815 r=0.6259 rho=0.5234 tau=0.3671
  [simple_avg3] RAE=0.6684 MAE=0.6081 R2=0.4920 r=0.7209 rho=0.6626 tau=0.4747

             RAE     MAE      R2  Pearson  Spearman  Kendall
ecfp4     0.7367  0.6703  0.3926   0.6279    0.5799   0.4069
scaffold  0.7795  0.7093  0.3089   0.5634    0.5018   0.3490
physchem  0.7074  0.6436  0.3815   0.6259    0.5234   0.3671
avg3      0.6684  0.6081  0.4920   0.7209    0.6626   0.4747


In [7]:
# ---- ElasticNet stacking of A, B, C ----
print("\n=== ElasticNet stacking ===", flush=True)
oof_stack = np.column_stack([oof_knn_a, oof_knn_b, oof_knn_c])
# Sweep gamma for physchem sim and find best stack
meta = ElasticNetCV(l1_ratio=[0.1,0.5,0.9,1.0], cv=5, max_iter=5000, random_state=SEED)
meta.fit(oof_stack, y_tr)
oof_stacked = meta.predict(oof_stack)
m_stk = full_metrics(y_tr, oof_stacked, cliff_pairs, "elasticnet_stack")
print(f"ElasticNet weights: A={meta.coef_[0]:.4f}  B={meta.coef_[1]:.4f}  C={meta.coef_[2]:.4f}")
print(f"Intercept: {meta.intercept_:.4f}")

# Nested CV stacking for unbiased OOF
print("\nNested-CV ElasticNet stacking...", flush=True)
oof_nested = np.full(len(y_tr), np.nan)
for k, (tr_idx, va_idx) in enumerate(splits):
    meta_tr_idx = [i for fold,(ti,_) in enumerate(splits) for i in ti if fold!=k]
    mn = ElasticNetCV(l1_ratio=[0.1,0.5,0.9,1.0], cv=5, max_iter=5000, random_state=SEED)
    mn.fit(oof_stack[meta_tr_idx], y_tr[meta_tr_idx])
    oof_nested[va_idx] = mn.predict(oof_stack[va_idx])
    print(f"  fold {k+1}  RAE={rae(y_tr[va_idx], oof_nested[va_idx]):.4f}")

m_nested = full_metrics(y_tr, oof_nested, cliff_pairs, "nested_stack")

# Best OOF: pick min RAE
best_oof_name, best_oof, best_m = "avg3", oof_avg3, m_avg
for name, arr, m in [("nested", oof_nested, m_nested), ("stacked", oof_stacked, m_stk)]:
    if m["RAE"] < best_m["RAE"]:
        best_oof_name, best_oof, best_m = name, arr, m
print(f"\nBest OOF: {best_oof_name}  RAE={best_m['RAE']:.4f}")



=== ElasticNet stacking ===


  [elasticnet_stack] RAE=0.6326 MAE=0.5756 R2=0.5284 r=0.7269 rho=0.6687 tau=0.4800
ElasticNet weights: A=0.5340  B=0.2510  C=0.5146
Intercept: -1.3003

Nested-CV ElasticNet stacking...


  fold 1  RAE=0.5475


  fold 2  RAE=0.6260


  fold 3  RAE=0.6935


  fold 4  RAE=0.6453


  fold 5  RAE=0.6796
  [nested_stack] RAE=0.6325 MAE=0.5755 R2=0.5287 r=0.7271 rho=0.6689 tau=0.4802

Best OOF: nested  RAE=0.6325


In [8]:
# ---- Final test predictions ----
print("\nComputing test predictions...", flush=True)

# System A: ECFP4
sim_a_te = tanimoto_sim(fps_te, fps_tr)
te_a = knn_predict(sim_a_te, y_tr)

# System B: Scaffold Tanimoto
sim_b_te = tanimoto_sim(scaf_fps_te, scaf_fps_tr)
te_b = knn_predict(sim_b_te, y_tr)

# System C: Physchem RBF
sim_c_te = physchem_sim(phys_arr_te_n, phys_arr_tr_n, gamma=0.5)
te_c = knn_predict(sim_c_te, y_tr)

# Final meta prediction on test
meta_final = ElasticNetCV(l1_ratio=[0.1,0.5,0.9,1.0], cv=5, max_iter=5000, random_state=SEED)
meta_final.fit(oof_stack, y_tr)
te_stack_input = np.column_stack([te_a, te_b, te_c])
te_stacked = meta_final.predict(te_stack_input)

# Use best strategy for test
te_avg3 = (te_a + te_b + te_c) / 3
if best_oof_name in ("nested", "stacked"):
    te_preds = te_stacked
else:
    te_preds = te_avg3

te_preds = np.clip(te_preds, y_tr.min()-0.5, y_tr.max()+0.5)

np.save(DATA_PROCESSED/"oof_scaffold_aware_knn.npy", best_oof)
np.save(DATA_PROCESSED/"te_oof_scaffold_aware_knn.npy", te_preds)
# Also save component OOFs for later stacking
np.save(DATA_PROCESSED/"oof_knn_ecfp4_only.npy",    oof_knn_a)
np.save(DATA_PROCESSED/"oof_knn_scaffold_only.npy", oof_knn_b)
np.save(DATA_PROCESSED/"oof_knn_physchem_only.npy", oof_knn_c)

sub = pd.DataFrame({"Molecule Name": te["name"].values, "pEC50": te_preds})
assert len(sub)==513 and sub["pEC50"].notna().all()
p = SUBMISSIONS/"98_scaffold_aware_knn.csv"; sub.to_csv(p, index=False)
print(f"Saved {p}")
print(f"Test: min={te_preds.min():.2f} med={np.median(te_preds):.2f} max={te_preds.max():.2f}")
print(f"\n*** nb98 OOF RAE = {best_m['RAE']:.4f} ***")



Computing test predictions...


Saved D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\98_scaffold_aware_knn.csv
Test: min=3.03 med=5.05 max=5.85

*** nb98 OOF RAE = 0.6325 ***
